<!-- nav -->
[← 2 · One value, a domain in each language](02_One_Value_A_Domain_In_Each_Language.ipynb) · [Home](../../README.md) · [4 · Names →](04_Names.ipynb)

# 3 · The smallest expression, a literal

Tutorial 2 read, by name, the domain each language's standard gives `42`. This one writes `42` as an expression, the
smallest there is: one literal. In every dialect a literal is a **term**, a value, held by an **expression**, a
reference object of the store; each language prints it as its own text, and Python's is read back from that text. Its
domain is inferred and its value evaluated, each by its language's rules. Then come the literals no scalar holds, such
as Python's `2j` and `None`, and a literal whose text holds no value, kept as written.

Every model here is written in the builders' fluent calls
([PRI-019: fluent builder syntax](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-019)), which show the model as the store holds it; the writers
Part II uses are shown once, in 03.1, as the shorthand they are. Part I starts from the toolkit's `dialects()`, a store
with every dialect installed and defined. `S` is its holder namespace, `store.Expressions`, and `amd64` x86-64's System V
ABI, the dialect C's terms are read under (02.3).

In [1]:
import assert from "node:assert/strict";

import * as Translators from "@mbse/expressions/Translators";
import * as C from "@mbse/expressions/C/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import { Terms } from "@mbse/expressions/Framework";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import { Simplifiers, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const amd64 = store.Instances.Expressions.SystemV.AMD64;

<a id="T03.1"></a>
## 03.1 · A term, held by an expression

An expression is a reference object, made in a transaction by its dialect's **holder**: Patterns' is
`S.Patterns.Expressions.Expression`. Its `root` holds a term, a value of the dialect's `Term` union, whose branch is its
kind, here `literal`, with its `value`, `42`. `forty_two` is that term's spec, the callable the builders call with the
union to write, so it writes `42` into any dialect's `Term` with a `literal` kind. `Terms.term` reads the term an
expression holds, `Terms.dialectOf` and `Terms.kindOf` its dialect and kind, and `Terms.form` its kind, its attributes
and its arguments, of which a literal has none:

In [2]:
const forty_two = (u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(42n))); // a spec: the term 42, in any dialect's Term
const answer = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root(forty_two)));
const term = Terms.term(answer);
const form = Terms.form(term);
console.log(answer.as_schema().name, "ref:", answer.as_schema().ref);
console.log(Terms.dialectOf(term), Terms.kindOf(term)[0]);
console.log(form.kind, Object.fromEntries(form.attributes), form.arguments.length);
assert(Terms.holderOf(term) === answer);
assert(form.kind === "literal" && form.attributes.size === 1 && form.attributes.get("value") === 42n && form.arguments.length === 0);

Expressions.Patterns.Expressions.Expression ref: true


Expressions.Patterns.Expressions.Term literal


literal { value: 42n } 0


Part II writes its terms with **writers** instead: each dialect's functions that write a term through these same
builders, a shorthand for the calls above. A writer writes the same term as the builder calls, which `Terms.same`
compares by structure (dialect, kind, attributes and arguments). Here are Patterns', Python3's, C's and SystemVerilog's:

In [3]:
const pairs: Record<string, [any, any]> = store.transaction(() => ({
  Patterns: [S.Patterns.Expressions.Expression((e: any) => e.root(E.literal(42n))), answer],
  Python3: [S.Python.Python3.Expression((e: any) => e.root(Python3.Expressions.constant(42n))),
    S.Python.Python3.Expression((e: any) => e.root(forty_two))],
  C: [S.C.Expressions.Expression((e: any) => e.root(C.Expressions.literal(42n)).abi([amd64])),
    S.C.Expressions.Expression((e: any) => e.root(forty_two).abi([amd64]))],
  SystemVerilog: [S.SystemVerilog.Expressions.Expression((e: any) => e.root(SystemVerilog.Expressions.literal(42n))),
    S.SystemVerilog.Expressions.Expression((e: any) => e.root(forty_two))],
}));
for (const [dialect, [written, built]] of Object.entries(pairs)) {
  console.log(dialect.padEnd(13), Terms.same(written, built));
  assert(Terms.same(written, built));
}

Patterns      true


Python3       true


C             true


SystemVerilog true


The rest of Part I writes builder calls, so that what you read is the model itself; Part II keeps its writers.

<a id="T03.2"></a>
## 03.2 · Read and printed

Python3's `42` is the same spec in its own holder, `S.Python.Python3.Expression`, and `Python3.Text.ToText` prints it as
Python source. A literal also keeps how it was spelled, beside its value: `0x2A` is the value 42, with its `text` as
written and the `base` that text encodes, 16, so it prints back as written
([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)). Python's notebook also reads `0x2A` with
`Python3.Text.FromText`, which gives the term built here; only Python reads Python source, with Python's own parser,
and TypeScript builds the same term
([Equivalence of the implementations](../../docs/EQUIVALENCE.md#deliberate-differences), "Reading Python source").

In [4]:
const [plain, hexadecimal] = store.transaction(() => [
  S.Python.Python3.Expression((e: any) => e.root(forty_two)),
  S.Python.Python3.Expression((e: any) => e.root(
    (u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(42n)).text("0x2A").base(16n))))]);
// Python reads it: `Python3.Text.FromText("0x2A")` gives the term `hexadecimal` holds
for (const expression of [plain, hexadecimal]) {
  console.log(Python3.Text.ToText(expression).padEnd(5), Object.fromEntries(Terms.form(Terms.term(expression)).attributes));
}
assert(Python3.Text.ToText(plain) === "42" && Python3.Text.ToText(hexadecimal) === "0x2A");

42    { value: 42n }


0x2A  { value: 42n, text: "0x2A", base: 16n }


<a id="T03.3"></a>
## 03.3 · The same literal in each language

Each language's dialect has its own holder and its own `Term`, whose `literal` kind refines the framework's, with what
its language's spelling encodes. TypeScript's `42` is a float, as every JavaScript number is: its value is `42.0`, its
`text` `42`. C's holder names the ABI its terms are read under, `amd64`. Each prints its own text, Excel's a formula:

In [5]:
const literals: Record<string, [any, any]> = store.transaction(() => ({
  TypeScript5: [S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(42.0)).text("42")))), TypeScript5.Text],
  C: [S.C.Expressions.Expression((e: any) => e.root(forty_two).abi([amd64])), C.Text],
  SystemVerilog: [S.SystemVerilog.Expressions.Expression((e: any) => e.root(forty_two)), SystemVerilog.Text],
  Excel: [S.Excel.Expressions.Expression((e: any) => e.root(forty_two)), Excel.Text],
  MATLAB: [S.MATLAB.Expressions.Expression((e: any) => e.root(forty_two)), Matlab.Text],
  LaTeX: [S.LaTeX.Expressions.Expression((e: any) => e.root(forty_two)), Latex.Text],
}));
for (const [dialect, [expression, text]] of Object.entries(literals)) {
  console.log(dialect.padEnd(13), text.ToText(expression).padEnd(4), Terms.dialectOf(Terms.term(expression)));
}
assert(Object.values(literals).map(([expression, text]) => text.ToText(expression)).join(" ") === "42 42 42 =42 42 42");

TypeScript5   42   Expressions.TypeScript.TypeScript5.Term


C             42   Expressions.C.Expressions.Term


SystemVerilog 42   Expressions.SystemVerilog.Expressions.Term


Excel         =42  Expressions.Excel.Expressions.Term


MATLAB        42   Expressions.MATLAB.Expressions.Term


LaTeX         42   Expressions.LaTeX.Expressions.Term


Each term holds its dialect, so none is another's: Python3's `42` and C's are not the same term, and C's dialect reads
no Python3 term:

In [6]:
const c = literals.C![0];
console.log(Terms.same(plain, c), C.Expressions.DIALECT.validate(plain));
assert(!Terms.same(plain, c) && C.Expressions.DIALECT.validate(plain).join() === "a Python3 literal is not a C term");

false [ "a Python3 literal is not a C term" ]


<a id="T03.4"></a>
## 03.4 · Its domain, inferred

Each dialect's `DIALECT.infer` gives the domain of a term's value: the domains tutorial 2 read by name, Python's `int`,
TypeScript's `number`, C's `int` under `amd64` and SystemVerilog's `integer`, 4-state (an unsized decimal literal is an
`integer`, not an `int`), and Excel's `number` and MATLAB's `double`. Patterns has no stored domains, by design: its
`int` is a native type's. C's `int` is the ABI's, not C's, so a C literal in a holder that names no ABI has no domain.
Inference is code until [CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md) makes it data:

In [7]:
const inferred: Record<string, [any, any]> = {
  Patterns: [E.DIALECT, answer],
  Python3: [Python3.Expressions.DIALECT, plain],
  TypeScript5: [TypeScript5.Expressions.DIALECT, literals.TypeScript5![0]],
  C: [C.Expressions.DIALECT, c],
  SystemVerilog: [SystemVerilog.Expressions.DIALECT, literals.SystemVerilog![0]],
  Excel: [Excel.Expressions.DIALECT, literals.Excel![0]],
  MATLAB: [Matlab.Expressions.DIALECT, literals.MATLAB![0]],
};
for (const [dialect, [reader, expression]] of Object.entries(inferred)) {
  console.log(dialect.padEnd(13), reader.infer(expression).name());
}
assert(Object.values(inferred).map(([reader, x]) => reader.infer(x).name()).join() === "int,int,number,int,integer,number,double");
const no_abi = store.transaction(() => S.C.Expressions.Expression((e: any) => e.root(forty_two)));
try {
  C.Expressions.DIALECT.infer(no_abi);
  throw new Error("inferred");
} catch (error) {
  assert((error as Error).name === "ValueError", String(error));
  console.log(String(error));
}

Patterns      int


Python3       int


TypeScript5   number


C             int


SystemVerilog integer


Excel         number


MATLAB        double


ValueError: cannot infer the domain of an invalid expression: 'int' is not a type C defines, and its expression names no ABI


<a id="T03.5"></a>
## 03.5 · Its value

Each dialect's `Evaluators.OfAny` evaluates a term by its language's rules: Patterns', Python's and C's `42` is an
integer; TypeScript's a float, as are Excel's and MATLAB's, whose numbers are doubles; SystemVerilog's a 4-state value of
its domain, `integer`, 32 bits signed, written as SystemVerilog writes it. LaTeX typesets, and evaluates nothing: it has
no evaluator.

In [8]:
const evaluated: Record<string, [any, any]> = {
  Patterns: [Simplifiers, answer],
  Python3: [Python3.Simplifiers, plain],
  TypeScript5: [TypeScript5.Simplifiers, literals.TypeScript5![0]],
  C: [C.Simplifiers, c],
  SystemVerilog: [SystemVerilog.Simplifiers, literals.SystemVerilog![0]],
  Excel: [Excel.Simplifiers, literals.Excel![0]],
  MATLAB: [Matlab.Simplifiers, literals.MATLAB![0]],
};
for (const [dialect, [evaluators, expression]] of Object.entries(evaluated)) {
  console.log(dialect.padEnd(13), String(evaluators.OfAny(expression)));
}
console.log("LaTeX has an evaluator:", "Simplifiers" in Latex);
assert(Object.values(evaluated).map(([evaluators, x]) => String(evaluators.OfAny(x))).join() ===
  "42,42,42,42,32'sb00000000000000000000000000101010,42,42");

Patterns      42


Python3       42


TypeScript5   42


C             42


SystemVerilog 32'sb00000000000000000000000000101010


Excel         42


MATLAB        42


LaTeX has an evaluator: false


<a id="T03.6"></a>
## 03.6 · 2.5, a string, a boolean

Other scalars are literals the same way. `Translators.between(source, target).forward(term)` translates a term to
another dialect (tutorial 14 tells how); here Patterns' `2.5`, `'hi'` and `True` in each language, a boolean spelled
six ways:

In [9]:
const Term = S.Framework.Term;
const targets: Record<string, [any, any]> = {
  Python3: [Python3.Expressions.DIALECT, Python3.Text], C: [C.Expressions.DIALECT, C.Text],
  SystemVerilog: [SystemVerilog.Expressions.DIALECT, SystemVerilog.Text],
  Excel: [Excel.Expressions.DIALECT, Excel.Text], MATLAB: [Matlab.Expressions.DIALECT, Matlab.Text],
  LaTeX: [Latex.Expressions.DIALECT, Latex.Text],
};
console.log(Object.keys(targets).map((dialect) => dialect.padEnd(16)).join("").trimEnd());
let texts: string[] = [];
for (const value of [2.5, "hi", true]) {
  const translated: any[] = store.transaction(() => {
    const patterns = S.Patterns.Expressions.Expression((e: any) => e.root(
      (u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(value)))));
    return Object.values(targets).map(([dialect]) => Term(Translators.between(E.DIALECT, dialect).forward(patterns)));
  });
  texts = translated.map((t, i) => Object.values(targets)[i]![1].ToText(t));
  console.log(texts.map((text) => text.padEnd(16)).join("").trimEnd());
}
assert(texts.join(" ") === "True true 1'b1 =TRUE true \\mathrm{true}");

Python3         C               SystemVerilog   Excel           MATLAB          LaTeX


2.5             2.5             2.5             =2.5            2.5             2.5


'hi'            "hi"            "hi"            ="hi"           "hi"            \text{hi}


True            true            1'b1            =TRUE           true            \mathrm{true}


A string's quotes are part of its spelling: Python's `'hi'` and `"hi"` are the same value but not the same term, since
`Terms.same` compares their `text` and `quote` too (Python's notebook also reads both with `FromText`):

In [10]:
const [single, double] = store.transaction(() => [
  S.Python.Python3.Expression((e: any) => e.root(
    (u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("hi")).text("'hi'").quote("'")))),
  S.Python.Python3.Expression((e: any) => e.root(
    (u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("hi")).text('"hi"').quote('"'))))]);
// Python reads them: `Python3.Text.FromText("'hi'")` and `Python3.Text.FromText('"hi"')` give these terms
console.log(Python3.Text.ToText(single), Python3.Text.ToText(double), "same term:", Terms.same(single, double),
  "same value:", Python3.Simplifiers.OfAny(single).evaluate() === Python3.Simplifiers.OfAny(double).evaluate());
assert(!Terms.same(single, double) && Python3.Simplifiers.OfAny(single).evaluate() === "hi" && Python3.Simplifiers.OfAny(double).evaluate() === "hi");

'hi' "hi" same term: false same value: true


<a id="T03.7"></a>
## 03.7 · What no scalar holds

A literal holds a value of a schema
([CAP-233: literals of any value](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-233.md)): a scalar, a container or a value object. What has
no schema is its dialect's own kind, `constant`, with its spelling too: Python's `2j`, a `complex` of two IEEE 754
`binary64` parts, and the singletons `None` and `...`; TypeScript's `null` and a regular expression, its `pattern` and
`flags`; C's `nullptr`. Each prints back as written
(Python's notebook also reads Python's three with `FromText`, as the terms built here):

In [11]:
const constants: Record<string, [any, any]> = store.transaction(() => ({
  "2j": [S.Python.Python3.Expression((e: any) => e.root(
    (u: any) => u.constant((k: any) => k.complex((z: any) => z.real(0.0).imag(2.0)).text("2j")))), Python3.Text],
  "None": [S.Python.Python3.Expression((e: any) => e.root(
    (u: any) => u.constant((k: any) => k.singleton("None").text("None")))), Python3.Text],
  "...": [S.Python.Python3.Expression((e: any) => e.root(
    (u: any) => u.constant((k: any) => k.singleton("...").text("...")))), Python3.Text],
  "null": [S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.constant((k: any) => k.singleton("null")))), TypeScript5.Text],
  "/a+/gu": [S.TypeScript.TypeScript5.Expression((e: any) => e.root(
    (u: any) => u.constant((k: any) => k.pattern("a+").flags("gu")))), TypeScript5.Text],
  "nullptr": [S.C.Expressions.Expression((e: any) => e.root(
    (u: any) => u.constant((k: any) => k.singleton("nullptr"))).abi([amd64])), C.Text],
}));
for (const [expression, text] of Object.values(constants)) {
  const form = Terms.form(Terms.term(expression));
  console.log(`${text.ToText(expression).padEnd(8)} ${Terms.dialectOf(Terms.term(expression)).padEnd(39)}`, form.kind,
    [...form.attributes.keys()]);
}
assert(Object.entries(constants).every(([source, [expression, text]]) => text.ToText(expression) === source));

2j       Expressions.Python.Python3.Term         constant [ "complex", "text" ]


None     Expressions.Python.Python3.Term         constant [ "singleton", "text" ]


...      Expressions.Python.Python3.Term         constant [ "singleton", "text" ]


null     Expressions.TypeScript.TypeScript5.Term constant [ "singleton" ]


/a+/gu   Expressions.TypeScript.TypeScript5.Term constant [ "pattern", "flags" ]


nullptr  Expressions.C.Expressions.Term          constant [ "singleton" ]


A `constant`'s domain is inferred, `complex`, `NoneType` or `ellipsis`, Python's names for its type, but Python3's
evaluator refuses to evaluate one, by name:

In [12]:
for (const source of ["2j", "None", "..."]) {
  const expression = constants[source]![0];
  try {
    Python3.Simplifiers.OfAny(expression).evaluate();
    throw new Error("evaluated");
  } catch (error) {
    assert((error as Error).name === "NotImplementedError", String(error));
    console.log(`${source.padEnd(4)} ${Python3.Expressions.DIALECT.infer(expression).name().padEnd(8)} ${String(error)}`);
  }
}

2j   complex  NotImplementedError: Python3 constant is not evaluated


None NoneType NotImplementedError: Python3 constant is not evaluated


...  ellipsis NotImplementedError: Python3 constant is not evaluated


<a id="T03.8"></a>
## 03.8 · A literal kept as written, valid or not

A literal's text may hold no value: `08` is no Python int, since Python 3 forbids leading zeros in a decimal literal. A
reader that keeps what it reads ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)) writes it as a literal with
its `text` and its `problems`, and no `value`. It prints back as written; validation reports its problems, and
inference and evaluation refuse it.

In [13]:
const invalid = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.literal(
  (r: any) => r.text("08").problems(["leading zeros in a decimal integer literal are not permitted"])))));
console.log(Python3.Text.ToText(invalid), [...Terms.form(Terms.term(invalid)).attributes.keys()]);
console.log(Python3.Expressions.DIALECT.validate(invalid));
for (const refused of [(x: any) => Python3.Expressions.DIALECT.infer(x), (x: any) => Python3.Simplifiers.OfAny(x).evaluate()]) {
  try {
    refused(invalid);
    throw new Error("not refused");
  } catch (error) {
    assert((error as Error).name === "ValueError", String(error));
    console.log(String(error));
  }
}
assert(Python3.Text.ToText(invalid) === "08");
assert(Python3.Expressions.DIALECT.validate(invalid).join() === "leading zeros in a decimal integer literal are not permitted");

08 [ "text", "problems" ]


[ "leading zeros in a decimal integer literal are not permitted" ]


ValueError: cannot infer the domain of an invalid expression: leading zeros in a decimal integer literal are not permitted


ValueError: a literal needs a value


## Why it works this way

- **A term is a value, its expression a reference object.** A literal is data its holder owns, compared by structure
  (`Terms.same`), never by identity; the expression gives it a place in the store, its dialect and, for C, its ABI.
- **A literal keeps its spelling beside its value**
  ([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)), so that text printed from a model is the text it
  was read from, and a text that holds no value is kept, with its problems
  ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)).
- **What no schema holds is a dialect's own kind** ([CAP-233: literals of any value](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-233.md)):
  Python's complex numbers and singletons, TypeScript's `null` and regular expressions, C's `nullptr`.
- **A domain is inferred, not stored on the term**, until
  [CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md); tutorial 4 goes on to names.

<!-- nav -->
[← 2 · One value, a domain in each language](02_One_Value_A_Domain_In_Each_Language.ipynb) · [Home](../../README.md) · [4 · Names →](04_Names.ipynb)